In [1]:
from src.align import preds_align

import numpy as np

from time import perf_counter


In [2]:
src = dict(np.load("calderon_output/1/all.npz"))
src.pop('model')
tgt = dict(np.load("calderon_output/8/all.npz"))
tgt.pop('model')

start = perf_counter()
aligner = preds_align.FitAffine().fit(src, tgt)
end = perf_counter()
print(f"{end - start:.4f}")

0.0940


In [3]:
import open3d as o3d
from src.align.align_utils import get_conf_mask
from src.align.align_utils import get_pointmap

def preds_to_pcd(preds, pointmap):
    mask = get_conf_mask(preds, lower_p=60, min_conf=1.005, upper_p=90)
    colors = preds['images'][mask].reshape(-1, 3)
    points = pointmap[mask].reshape(-1, 3)
    points[:, 0] *= -1
    points[:, 1] *= -1
    
    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(points)
    point_cloud.colors = o3d.utility.Vector3dVector(colors)
    return point_cloud

In [4]:
src_pcd = preds_to_pcd(src, aligner.transform(src))
#src_pcd = preds_to_pcd(src, get_pointmap(src))
tgt_pcd = preds_to_pcd(tgt, get_pointmap(tgt))

In [5]:
#o3d.visualization.draw_geometries([src_pcd, tgt_pcd])

In [6]:
preds = [
    dict(np.load(f"calderon_output/{i}/all.npz"))
    for i in range(1, 9)
]

for p in preds:
    p.pop('model')

In [7]:
from src.align.homography import transforms

measures = []
for tgt, src in zip(preds[:-1], preds[1:]):
    measures.append(preds_align.FitAffine().fit(src, tgt))
measures.append(preds_align.FitAffine().fit(preds[0], preds[-1]))

In [8]:
current_est = preds_align.FitAffine()
current_est._transform = transforms.Affine.identity()
estimations = [current_est]
for meas in measures[:-1]:
    current_est = current_est@meas
    estimations.append(current_est)

In [9]:
#pcd = [
#    preds_to_pcd(p, est.transform(p))
#    for p, est in zip(preds, estimations)
#]
#o3d.visualization.draw_geometries(pcd)

In [9]:
from src.align.homography.graph.core import Vertex, Optimizer
from src.align.homography.graph.edges import EdgeSL4Affine
from src.align.homography.graph.algorithms import GaussNewton

In [10]:
vertices = [
    Vertex.Affine(i, est._transform.copy())
    for i, est in enumerate(estimations)
]

In [11]:
Edges = [
    EdgeSL4Affine(v_parent, v_child, meas._transform.copy())
    for (v_parent, v_child), meas in zip(zip(vertices[:-1], vertices[1:]), measures[:-1])
]
Edges.append(EdgeSL4Affine(vertices[-1], vertices[0], measures[-1]._transform.copy()))

In [12]:
for e in Edges:
    print(e.parent.idx, e.child.idx, e.transform)

0 1 Affine(mat: [ 0.97770166 -0.15074874 -0.11831456 35.27751851  0.0668835   0.84985825
 -0.51737508  7.91286667  0.18033139  0.50213705  0.8377068  33.81945221
  0.          0.          0.          1.        ])
1 2 Affine(mat: [ 0.38597949 -0.22896573 -0.95815816  8.2914812   0.61084494  0.80006947
  0.06591346 28.66390366  0.70194771 -0.57042061  0.46750499 32.72819611
  0.          0.          0.          1.        ])
2 3 Affine(mat: [-0.23394092 -0.30183763 -1.0814245  30.28906108  0.30382672  1.07092864
 -0.34617222 12.01837502  1.10301636 -0.35890921 -0.13447611 46.91621858
  0.          0.          0.          1.        ])
3 4 Affine(mat: [ 8.90134669e-01  3.57741735e-02  2.70623809e-01  4.89423923e+01
 -4.62623652e-02  9.38756986e-01  2.98994014e-02  4.62978155e+00
 -2.73750283e-01 -4.23224816e-02  8.74358261e-01  3.02857615e+01
  0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00])
4 5 Affine(mat: [ -0.50588933  -0.23792618  -0.91936174 -11.10308073   0.23766386
 

In [13]:
for v in vertices:
    print(v.estimate)

Affine(mat: [1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1.])
Affine(mat: [ 0.97770166 -0.15074874 -0.11831456 35.27751851  0.0668835   0.84985825
 -0.51737508  7.91286667  0.18033139  0.50213705  0.8377068  33.81945221
  0.          0.          0.          1.        ])
Affine(mat: [ 0.20223805 -0.27698058 -1.00204184 35.19084414  0.18177702  0.95975302
 -0.24994331 15.89493191  0.96435846 -0.11739041  0.25194372 77.12450703
  0.          0.          0.          1.        ])
Affine(mat: [-1.23673439e+00  1.97256317e-03  1.19284872e-02 -9.02442552e+00
 -2.66180333e-02  1.06266680e+00 -4.95206554e-01  2.12090642e+01
  1.66287922e-02 -5.07221339e-01 -1.03612398e+00  1.16743424e+02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00])
Affine(mat: [-1.10421684e+00 -4.28962364e-02 -3.24201021e-01 -6.91827694e+01
  6.27078203e-02  1.01759202e+00 -4.08418313e-01  9.82852150e+00
  3.21906356e-01 -4.31711356e-01 -9.16609026e-01  8.38291491e+01
  0.00000000e+00  0.00000000e+00  0.00

In [14]:
for est in estimations:
    print(est._transform)

Affine(mat: [1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1.])
Affine(mat: [ 0.97770166 -0.15074874 -0.11831456 35.27751851  0.0668835   0.84985825
 -0.51737508  7.91286667  0.18033139  0.50213705  0.8377068  33.81945221
  0.          0.          0.          1.        ])
Affine(mat: [ 0.20223805 -0.27698058 -1.00204184 35.19084414  0.18177702  0.95975302
 -0.24994331 15.89493191  0.96435846 -0.11739041  0.25194372 77.12450703
  0.          0.          0.          1.        ])
Affine(mat: [-1.23673439e+00  1.97256317e-03  1.19284872e-02 -9.02442552e+00
 -2.66180333e-02  1.06266680e+00 -4.95206554e-01  2.12090642e+01
  1.66287922e-02 -5.07221339e-01 -1.03612398e+00  1.16743424e+02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00])
Affine(mat: [-1.10421684e+00 -4.28962364e-02 -3.24201021e-01 -6.91827694e+01
  6.27078203e-02  1.01759202e+00 -4.08418313e-01  9.82852150e+00
  3.21906356e-01 -4.31711356e-01 -9.16609026e-01  8.38291491e+01
  0.00000000e+00  0.00000000e+00  0.00

In [15]:
optim = Optimizer(
    GaussNewton,
    [v for v in vertices],
    [e for e in Edges]
)

In [16]:
optim.optimize(10)

1. Loss: 277.68301391641876. d.norm: 34.13193893432617
2. Loss: 0.2161300324369222. d.norm: 1.0859086513519287
3. Loss: 0.018337076995521784. d.norm: 0.031497642397880554
4. Loss: 0.018334189779125154. d.norm: 0.0025017051957547665
5. Loss: 0.018334180233068764. d.norm: 0.00014994390949141234
6. Loss: 0.018334171269088984. d.norm: 3.758936145459302e-05
7. Loss: 0.018334175227209926. d.norm: 1.4625705262005795e-05
8. Loss: 0.018334181513637304. d.norm: 3.880075382767245e-05
9. Loss: 0.018334182328544557. d.norm: 4.366354914964177e-05
10. Loss: 0.01833418314345181. d.norm: 2.054044307442382e-05


In [17]:
new_aligners = []
for v in optim.vertices:
    aligner = preds_align.FitAffine()
    aligner._transform = v.estimate.copy()
    print(aligner._transform)
    new_aligners.append(aligner)

Affine(mat: [1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1.])
Affine(mat: [ 0.95225864 -0.16780274 -0.13693038 35.27762986  0.06965642  0.85448189
 -0.5085609   7.91296601  0.20950016  0.50528059  0.82378626 33.81949892
  0.          0.          0.          1.        ])
Affine(mat: [ 0.12856315 -0.2674206  -0.94444382 33.88202462  0.1855329   0.94284258
 -0.24219252 16.33913229  0.95913317 -0.11816308  0.18184519 77.00097
  0.          0.          0.          1.        ])
Affine(mat: [-1.13338325e+00  2.64987942e-02  1.22494327e-01 -9.74746005e+00
 -3.11412454e-02  1.04310222e+00 -4.79376674e-01  2.19275556e+01
 -9.52380083e-02 -4.80150987e-01 -1.00869860e+00  1.13163609e+02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00])
Affine(mat: [-1.05514234e+00 -1.16751960e-02 -1.64390940e-01 -6.13852990e+01
  4.28671563e-02  1.00097529e+00 -3.99722442e-01  1.07145924e+01
  1.81617249e-01 -4.12360935e-01 -9.45045584e-01  7.57302879e+01
  0.00000000e+00  0.00000000e+00  0.00000

In [18]:
new_pcd = [
    preds_to_pcd(p, est.transform(p))
    for p, est in zip(preds, new_aligners)
]

In [19]:
o3d.visualization.draw_geometries(new_pcd)

In [24]:
raise RuntimeError("Hasta aquí")

RuntimeError: Hasta aquí

In [20]:
all_points = np.vstack([
    pc.points for pc in new_pcd
])

In [21]:
centroid = np.mean(all_points, axis=0)
print("Centroid:", centroid)

Centroid: [ 6.18892896 -7.09727747 55.90294156]


In [22]:
# Estimate plane via PCA
points_mean = np.mean(all_points, axis=0)
cov = np.cov(all_points.T)
eig_vals, eig_vecs = np.linalg.eig(cov)
normal = eig_vecs[:, np.argmin(eig_vals)]  # Eigenvector with smallest eigenvalue

print("Plane normal:", normal)

Plane normal: [ 0.03825432 -0.97791748 -0.20546049]


In [24]:
import math

rotation_axis = normal  # Already normalized from PCA
angle_per_frame = np.radians(1)  # 1 degree per frame

In [25]:
combined_pcd = o3d.geometry.PointCloud()

# Concatenate all points (and optionally colors)
for pc in new_pcd:
    combined_pcd += pc

In [26]:
vis = o3d.visualization.Visualizer()
vis.create_window()
vis.add_geometry(combined_pcd)

# Move the geometry to the origin
combined_pcd.translate(-centroid)  # Temporarily center at origin

PointCloud with 9180646 points.

In [ ]:
import time

# Rotation matrix around an arbitrary axis using Rodrigues formula
def rotation_matrix(axis, theta):
    axis = axis / np.linalg.norm(axis)
    a = np.cos(theta / 2)
    b, c, d = -axis * np.sin(theta / 2)
    return np.array([
        [a*a + b*b - c*c - d*d, 2*(b*c - a*d),     2*(b*d + a*c)],
        [2*(b*c + a*d),     a*a + c*c - b*b - d*d, 2*(c*d - a*b)],
        [2*(b*d - a*c),     2*(c*d + a*b),     a*a + d*d - b*b - c*c]
    ])

try:
    while True:
        R = rotation_matrix(rotation_axis, -2*angle_per_frame)
        combined_pcd.rotate(R, center=(0,0,0))  # Rotate around centroid (centered at origin)
        vis.update_geometry(combined_pcd)
        vis.poll_events()
        vis.update_renderer()
        time.sleep(0.01)  # Controls speed
except KeyboardInterrupt:
    pass